# A randomized controlled trial of an intervention to reduce stigma toward people with opioid use disorder among primary care clinicians

<br>
Stephanie A. Hooker, A. Lauren Crain, Amy B. LaFrance, Sheryl Kane, J. Konadu Fokuo, Gavin Bart and Rebecca C. Rossom

Data archived at NIDA Data Share and accessible through the HEAL Data Platform.

## Replicate Original Analysis

### Install and import necessary software libraries

In [1]:
!pip install matplotlib -q
!pip install scikit-learn -q

import zipfile
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import sklearn
from sklearn import metrics
import os

### Retrieve and validate dataset

Here we retrieve the dataset from NIDA Data Share through the HEAL Data Platform. We also validate the dataset against the corresponding variable-level metadata, demonstrating that the data are consistent with the variable definitions.

In [2]:
if not os.path.exists('stigma_supp.csv'):
    os.system('gen3 drs-pull object dg.H34L/a6c7fc23-3534-401c-88ca-9fbbb495dcf4')
    with zipfile.ZipFile('ascii-crf-data-files_nida-ctn-0095a2 v1-1.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

In [3]:
!frictionless validate --schema schemas/stigma_supp.json --schema-sync stigma_supp.csv

─────────────────────────────────── Dataset ────────────────────────────────────
                     dataset                      
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ name        ┃ type  ┃ path            ┃ status ┃
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ stigma_supp │ table │ stigma_supp.csv │ VALID  │
└─────────────┴───────┴─────────────────┴────────┘


### Define functions for use in data transformation and analysis

In [4]:
def clean_df(df):

    col_names = {'DDBS': 'Overall Stigma', 'DDBS_DIFF': 'Difference', 'DDBS_DISDAIN': 'Disdain', 'DDBS_BLAME2': 'Blame', 
              'INTEND_WAIVER': 'Intentions to get waivered', 'INTEND_PRESCRIBE': 'Intentions to prescribe buprenorphine', 
              'WILLWORK': 'Willingness to work with OUD', 'TX_EFFECTIVE': 'Perceived OUD treatment effectiveness', 
              'TX_ADHERENCE': 'Perceived OUD treatment adherence', 'PCC_AGE': 'Age', 'PCC_GENDER': 'Gender', 'PCC_RACE': 'Race', 
              'PCC_HISPANIC': 'Ethnicity', 'STIGMA_GROUP': 'Stigma Group', 'PCC_WAIVERED': 'Waivered to prescribe buprenorphine', 'PCC_MD': 'Degree'}
    df.rename(columns=col_names, inplace=True)
    for field in ['STIG_CLIN_ID', 'HLTH_SYS_ID', 'PCC_ID', 'COMPLETE_TRAINING', 'COMPLETE_SURVEY']:
        if field in df.columns:
            df.drop([field], axis=1, inplace=True)

    df.Gender = df.Gender.fillna(7.0)
    df.Ethnicity = df.Ethnicity.fillna(7.0)
    df['Race'] = df.Race.map({'5 White': 'White', '9 Prefer not to answer': 'Prefer not to answer', '2 Asian': 'Asian', '3 Black or African American': 'Black or African American', 
                              '7 Multiple selected': 'Multiple selected', '6 Some other race': 'Some other race'})
    df['Gender'] = df.Gender.map({2.0: 'Female', 1.0: 'Male', 6.0: 'Not Listed', 7.0: 'Prefer not to answer'})
    df['Ethnicity'] = df.Ethnicity.map({0.0: 'Not Hispanic or Latino', 1.0: 'Hispanic or Latino',  7.0: 'Missing'})
    df['Waivered to prescribe buprenorphine'] = df['Waivered to prescribe buprenorphine'].map({0: 'False', 1: 'True'})
    df['Degree'] = df.Degree.map({1: 'MD/DO', 0: 'PA/NP'})

    keep_list = list(set(df.columns) & set(['Stigma Group', 'Overall Stigma', 'Difference', 'Disdain', 'Blame',  
                                                  'Intentions to get waivered', 'Intentions to prescribe buprenorphine', 
                                                  'Willingness to work with OUD', 'Perceived OUD treatment effectiveness', 
                                                  'Perceived OUD treatment adherence', 'Waivered to prescribe buprenorphine', 
                                                  'Gender', 'Ethnicity', 'Race', 'Degree']))
    df = df[keep_list]

    return df

def make_print_df(df):

    features = ['Gender', 'Ethnicity', 'Race', 'Waivered to prescribe buprenorphine', 'Degree']
    all_subj = []
    print_df = pd.DataFrame()

    N_AC = int(df['Stigma Group'].value_counts()['STIG_CTRL'])
    N_SR = int(df['Stigma Group'].value_counts()['STIG_INT'])
    N = N_AC + N_SR

    for feat in features:
        print_df1 = pd.DataFrame({'Category':  f'{feat} - ' + df[feat].value_counts().index, f'All N={N}': (100*df[feat].value_counts()/len(df)).values.round(2)} )
        print_df2 = pd.DataFrame({'Category': f'{feat} - ' + (100*df[df['Stigma Group'] == 'STIG_INT'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_INT'])).index, f'Stigma reduction n = {N_SR}': (100*df[df['Stigma Group'] == 'STIG_INT'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_INT'])).values.round(2)} )
        print_df3 = pd.DataFrame({'Category':  f'{feat} - ' + (100*df[df['Stigma Group'] == 'STIG_CTRL'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_CTRL'])).index, f'Attention-control n = {N_AC}':  (100*df[df['Stigma Group'] == 'STIG_CTRL'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_CTRL'])).values.round(2)} )

        print_df1 = print_df1.merge(print_df2, how='left', on='Category')
        print_df1 = print_df1.merge(print_df3, how='left', on='Category')
        print_df1.fillna(0, inplace=True)
        print_df1[[f'All N={N}', f'Stigma reduction n = {N_SR}', f'Attention-control n = {N_AC}']] = print_df1[[f'All N={N}', f'Stigma reduction n = {N_SR}', f'Attention-control n = {N_AC}']].astype(str)
        print_df1[f'All N={N}'] = print_df1[f'All N={N}'].apply( lambda x : str(x) + '%')
        print_df1[f'Stigma reduction n = {N_SR}'] = print_df1[f'Stigma reduction n = {N_SR}'].apply( lambda x : str(x) + '%')
        print_df1[f'Attention-control n = {N_AC}'] = print_df1[f'Attention-control n = {N_AC}'].apply( lambda x : str(x) + '%')
        print_df = pd.concat([print_df, print_df1])

        if feat != 'Degree':
            print_df = pd.concat([print_df,  pd.DataFrame({'Category': ['-'], f'All N={N}': ['-'], f'Stigma reduction n = {N_SR}': ['-'], f'Attention-control n = {N_AC}': ['-'] })])


    return print_df

def make_impact_df(df, features):
    stig_int = []
    stig_ctrl = []
    t_vals = []
    p_vals = []
    d_vals = []

    def cohen_d(x,y):
        nx = len(x)
        ny = len(y)
        dof = nx + ny - 2
        return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1) ** 2 + (ny-1)*np.std(y, ddof=1) ** 2) / dof)

    for feat in features:
        x = df[df['Stigma Group'] == 'STIG_INT'][feat].dropna()
        y = df[df['Stigma Group'] == 'STIG_CTRL'][feat].dropna()

        # Perform the t-test
        t_statistic, p_value = scipy.stats.ttest_ind(x, y)
        d = cohen_d(x, y).round(2)

        stig_int.append(f'{x.mean().round(1)} ({round(x.std(),1)})')
        stig_ctrl.append(f'{y.mean().round(1)} ({round(y.std(),1)})')
        t_vals.append(t_statistic)
        p_vals.append(p_value)
        d_vals.append(d)

    impact_df = pd.DataFrame({"Feature": features, 
                        "Stigma Reduction Mean (SD)": stig_int,
                        "Attention Control Mean (SD)": stig_ctrl,
                        "t-statistic": np.round(t_vals, 3),
                        "p-value": np.round(p_vals, 3),
                        "Cohen's d-value": d_vals})

    return impact_df

## Read, clean and inspect the dataset

In [5]:
df = pd.read_csv('stigma_supp.csv')
df = clean_df(df)
df.head()

,Difference,Blame,Stigma Group,Waivered to prescribe buprenorphine,Gender,Degree,Willingness to work with OUD,Race,Intentions to prescribe buprenorphine,Intentions to get waivered,Perceived OUD treatment effectiveness,Disdain,Ethnicity,Perceived OUD treatment adherence,Overall Stigma
0,5.33,5.5,STIG_CTRL,False,Male,MD/DO,2.00,White,3.0,1.0,2.0,6.33,Not Hispanic or Latino,2,5.75
1,3.33,5.0,STIG_INT,False,Male,MD/DO,3.00,White,2.0,2.0,3.0,5.33,Not Hispanic or Latino,3,4.50
2,4.33,5.0,STIG_CTRL,False,Male,MD/DO,3.00,White,3.0,2.0,3.0,5.33,Not Hispanic or Latino,2,4.88
3,3.67,3.5,STIG_INT,False,Female,MD/DO,2.67,White,2.0,2.0,2.0,4.67,Not Hispanic or Latino,3,4.00
4,4.33,3.0,STIG_INT,False,Female,PA/NP,2.67,White,3.0,3.0,3.0,5.00,Not Hispanic or Latino,2,4.25


### Replicate Table 1

Demographic characteristics of PCCs who completed stigma reduction or attention-control training

In [6]:
print_df = make_print_df(df)
print_df

,Category,All N=85,Stigma reduction n = 46,Attention-control n = 39
0,Gender - Female,57.65%,58.7%,56.41%
1,Gender - Male,36.47%,36.96%,35.9%
2,Gender - Prefer not to answer,4.71%,2.17%,7.69%
3,Gender - Not Listed,1.18%,2.17%,0.0%
0,-,-,-,-
0,Ethnicity - Not Hispanic or Latino,94.12%,97.83%,89.74%
1,Ethnicity - Hispanic or Latino,3.53%,2.17%,5.13%
2,Ethnicity - Missing,2.35%,0.0%,5.13%
0,-,-,-,-
0,Race - White,67.06%,63.04%,71.79%


### Replicate Table 2

Effect of stigma reduction vs. attention-control training on self-reported stigma and intentions to treat people with OUD

In [7]:
features = ['Overall Stigma', 'Difference', 'Disdain', 'Blame',  'Intentions to get waivered', 
                'Intentions to prescribe buprenorphine', 'Willingness to work with OUD', 'Perceived OUD treatment effectiveness', 'Perceived OUD treatment adherence']
impact_df = make_impact_df(df, features)
impact_df

,Feature,Stigma Reduction Mean (SD),Attention Control Mean (SD),t-statistic,p-value,Cohen's d-value
0,Overall Stigma,4.1 (1.3),4.2 (1.2),-0.481,0.632,-0.10
1,Difference,3.4 (1.7),3.1 (1.7),0.741,0.461,0.16
2,Disdain,4.7 (1.4),4.9 (1.4),-0.859,0.393,-0.19
3,Blame,4.4 (1.6),4.8 (1.6),-1.286,0.202,-0.28
4,Intentions to get waivered,2.3 (0.7),2.1 (0.8),1.113,0.269,0.26
5,Intentions to prescribe buprenorphine,3.2 (1.0),3.0 (0.9),0.899,0.372,0.21
6,Willingness to work with OUD,3.0 (0.7),3.1 (0.9),-0.828,0.410,-0.18
7,Perceived OUD treatment effectiveness,2.6 (0.8),2.7 (0.7),-0.745,0.459,-0.16
8,Perceived OUD treatment adherence,2.5 (0.6),2.4 (0.6),0.150,0.881,0.03


### Replicate Table 3

Correlations among Self-Reported Stigma and Intentions to Treat People with OUD (N = 85)

In [8]:
corr_matrix = df[features].corr().round(2)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_matrix.columns = ['1', '2', '3', '4', '5', '6', '7', '8', '9']
corr_matrix.mask(mask)

,1,2,3,4,5,6,7,8,9
Overall Stigma,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Difference,0.84,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Disdain,0.79,0.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Blame,0.64,0.31,0.33,NaN,NaN,NaN,NaN,NaN,NaN
Intentions to get waivered,-0.25,-0.06,-0.23,-0.35,NaN,NaN,NaN,NaN,NaN
Intentions to prescribe buprenorphine,-0.25,-0.11,-0.18,-0.35,0.61,NaN,NaN,NaN,NaN
Willingness to work with OUD,-0.40,-0.34,-0.29,-0.34,0.42,0.46,NaN,NaN,NaN
Perceived OUD treatment effectiveness,-0.32,-0.19,-0.26,-0.38,0.21,0.29,0.46,NaN,NaN
Perceived OUD treatment adherence,-0.39,-0.28,-0.31,-0.35,0.17,0.22,0.36,0.62,NaN


## Adding Data from Another Study (Managed Locally and Uploaded to Workspace)

### Validate new data

Here we validate the new dataset against the variable level metadata from the original dataset, ensuring they are harmonized; if problems were found, we could perform the necessary translation(s) here and check again.

In [9]:
!frictionless validate --schema schemas/stigma_supp.json --schema-sync data/synthetic_data.csv

─────────────────────────────────── Dataset ────────────────────────────────────
                           dataset                           
┏━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ name           ┃ type  ┃ path                    ┃ status ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ synthetic_data │ table │ data/synthetic_data.csv │ VALID  │
└────────────────┴───────┴─────────────────────────┴────────┘


### Read in new data and concatenate with original dataset

After concatenating the two datasets, we see that we now have 125 observations instead of the original 85.

In [10]:
synth_df = pd.read_csv('data/synthetic_data.csv')
synth_df = clean_df(synth_df)
cols = synth_df.columns
combined_df = pd.concat([df[synth_df.columns], synth_df])
print(f'{len(combined_df)} observations')

125 observations


### Repeat Table 1 with combined data

In [11]:
combined_print_df = make_print_df(combined_df)
combined_print_df

,Category,All N=125,Stigma reduction n = 63,Attention-control n = 62
0,Gender - Female,56.8%,55.56%,58.06%
1,Gender - Male,39.2%,41.27%,37.1%
2,Gender - Prefer not to answer,3.2%,1.59%,4.84%
3,Gender - Not Listed,0.8%,1.59%,0.0%
0,-,-,-,-
0,Ethnicity - Not Hispanic or Latino,84.0%,87.3%,80.65%
1,Ethnicity - Hispanic or Latino,14.4%,12.7%,16.13%
2,Ethnicity - Missing,1.6%,0.0%,3.23%
0,-,-,-,-
0,Race - White,53.6%,53.97%,53.23%


### Repeat Table 2 with combined data

In [12]:
features = ['Overall Stigma', 'Difference', 'Disdain', 'Blame', 'Intentions to get waivered']
combined_impact_df = make_impact_df(combined_df, features)
combined_impact_df

,Feature,Stigma Reduction Mean (SD),Attention Control Mean (SD),t-statistic,p-value,Cohen's d-value
0,Overall Stigma,4.1 (1.4),4.1 (1.5),0.089,0.930,0.02
1,Difference,3.5 (1.8),3.6 (1.8),-0.188,0.851,-0.03
2,Disdain,4.7 (1.6),4.8 (1.6),-0.340,0.735,-0.06
3,Blame,4.4 (1.8),4.8 (1.9),-1.075,0.284,-0.19
4,Intentions to get waivered,2.3 (0.8),2.6 (1.0),-1.356,0.178,-0.25
